# Enhanced Bone Fracture Detection: Hybrid CNN-ViT with Attention

## 1. Architectural Pipeline
As specified in the research paper, the model follows this pipeline:
1. **Pre-processing**: Resize to 224x224, normalize to [0,1], and apply medical-grade augmentation.
2. **CNN Branch (Local)**: Extracts spatial features (edges/textures) using a ResNet backbone.
3. **ViT Branch (Global)**: Captures long-range dependencies using a Vision Transformer.
4. **Fusion & Attention**: 
   - Spatially align CNN and ViT features.
   - Concatenate along the channel dimension.
   - Apply **Channel Attention** to weight important feature maps.
   - Apply **Spatial Attention** to focus on potential fracture sites.
5. **Classification**: Binary Cross-Entropy optimized with Adam.
6. **Explainability**: Grad-CAM heatmaps for clinical validation.

## 2. Environment & Data Setup

In [ ]:
import os
repo_name = "Enhanced-bone-fracture-detection"
if not os.path.exists(repo_name):
    !git clone https://github.com/Kaushikj-7/Enhanced-bone-fracture-detection.git

%cd {repo_name}
!pip install -r requirements.txt
!pip install kagglehub

In [ ]:
from src.dataset import get_dataloaders
data_dir = "data"
dataloaders, datasets = get_dataloaders(data_dir, batch_size=32, num_workers=2)
print(f"Dataset Ready: {len(datasets['train'])} training images.")

## 3. Stage 1: Feasibility Test (Sample Epoch)
Before full training, we run a small sample to verify that the gradient flow, attention weights, and loss computation are functioning correctly.

In [ ]:
# Running a small feasibility check (1 epoch, small subset if possible or just 1 full epoch)
!python main.py --experiments hybrid --epochs 1 --batch_size 16 --output_dir "outputs/test_run"

## 4. Stage 2: Production Pipeline (Full Training)
Once the test run is verified (proper loss reduction and no crashes), execute the full production pipeline.

In [ ]:
# Production training as per paper specifications
!python main.py --experiments cnn,vit,hybrid --epochs 20 --batch_size 32 --output_dir "outputs/production"

## 5. Inference & Clinical Visualization
Generate results for the best-performing model.

In [ ]:
import glob
from IPython.display import Image, display

print("Visualizing Hybrid Model Grad-CAM Results...")
results = glob.glob("outputs/production/hybrid/*.png")
for img_path in results[:3]: # Display first 3 results
    print(f"Result: {img_path}")
    display(Image(img_path))